In [25]:
import pandas as pd
from pathlib import Path
import numpy as np

## Path configuration

In [26]:
ROOT_DIR = Path().cwd().parent
DATASET_DIR = ROOT_DIR / "dataset"
CSV_FILE_PATH = DATASET_DIR / "raw" / "athlete_events.csv"
OUT_DIR=DATASET_DIR / "processed" 
OUT_CSV_FILE_PATH = OUT_DIR / "clean_athlete_events.csv"
print(ROOT_DIR,"\n",DATASET_DIR,"\n",CSV_FILE_PATH,"\n",OUT_CSV_FILE_PATH)

/home/mdev/Documents/ml/olympic_medals_prediction 
 /home/mdev/Documents/ml/olympic_medals_prediction/dataset 
 /home/mdev/Documents/ml/olympic_medals_prediction/dataset/raw/athlete_events.csv 
 /home/mdev/Documents/ml/olympic_medals_prediction/dataset/processed/clean_athlete_events.csv


## Load and Inspect

In [27]:
df = pd.read_csv(CSV_FILE_PATH)
df.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [28]:
rows,columns = df.shape
print(f"Rows: {rows}\nColumns: {columns}")

Rows: 271116
Columns: 15


In [29]:
column_names = df.columns.tolist()
column_names

['ID',
 'Name',
 'Sex',
 'Age',
 'Height',
 'Weight',
 'Team',
 'NOC',
 'Games',
 'Year',
 'Season',
 'City',
 'Sport',
 'Event',
 'Medal']

In [30]:
years = df['Year'].unique().tolist()
print(f"Years: {len(years)} \nSamples: {years[:10]}")

Years: 35 
Samples: [1992, 2012, 1920, 1900, 1988, 1994, 1932, 2002, 1952, 1980]


In [31]:
nations = df['NOC'].unique().tolist()
print(f"Nations: {len(nations)}\nSamples:{nations[:10]}")

Nations: 230
Samples:['CHN', 'DEN', 'NED', 'USA', 'FIN', 'NOR', 'ROU', 'EST', 'FRA', 'MAR']


In [32]:
sports = df['Sport'].unique().tolist()
print(f"Sports: {len(sports)}\nSamples:{sports[:10]}")

Sports: 66
Samples:['Basketball', 'Judo', 'Football', 'Tug-Of-War', 'Speed Skating', 'Cross Country Skiing', 'Athletics', 'Ice Hockey', 'Swimming', 'Badminton']


In [33]:
events = df['Event'].unique().tolist()
print(f"Events: {len(events)}\nSamples:{events[:10]}")

Events: 765
Samples:["Basketball Men's Basketball", "Judo Men's Extra-Lightweight", "Football Men's Football", "Tug-Of-War Men's Tug-Of-War", "Speed Skating Women's 500 metres", "Speed Skating Women's 1,000 metres", "Cross Country Skiing Men's 10 kilometres", "Cross Country Skiing Men's 50 kilometres", "Cross Country Skiing Men's 10/15 kilometres Pursuit", "Cross Country Skiing Men's 4 x 10 kilometres Relay"]


In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   39783 non-null   str    
dtypes: float64(3), int64(2), str(10)
memory usage: 31.0 MB


## Impute missing value age,height,weight

In [35]:
for col in ['Age','Height','Weight']:
    print(f"{col} Missing",df[col].isna().sum())
    df[col] = df[col].fillna(df[col].median())
df[['Age','Height','Weight']].isna().sum()

Age Missing 9474
Height Missing 60171
Weight Missing 62875


Age       0
Height    0
Weight    0
dtype: int64

## Feature extraction

In [36]:
new_cols = ['noc','season','year','athletes','athletes_female_percentage','avg_age','agv_height','avg_weight','prev_medals','prev_gold_medals','sports','events','total_medals']
print(f"Count: {len(new_cols)}\nList: {new_cols}")

Count: 13
List: ['noc', 'season', 'year', 'athletes', 'athletes_female_percentage', 'avg_age', 'agv_height', 'avg_weight', 'prev_medals', 'prev_gold_medals', 'sports', 'events', 'total_medals']


In [37]:
dataset = []

In [38]:
# sorted_years = np.sort(years)
# for noc in nations:
#     ds = []
#     for i,y in enumerate(sorted_years):
#         series = df[(df['NOC'] == noc) & (df['Year'] == y)]
        
#         if series.empty:
#             continue

#         season = series['Season'].iloc[0]
#         athletes = series['ID'].unique().tolist()
#         female_percentage = series['Sex'].value_counts(normalize=True).to_dict().get('F',0.0)
#         avg_age = np.ceil(series['Age'].mean())
#         avg_height = np.ceil(series['Height'].mean())
#         avg_weight = np.ceil(series['Weight'].mean())

#         prev_year = sorted_years[i-1] if i != 0 else np.nan
#         last_year_records = df[(df['NOC'] == noc) & (df['Year'] == prev_year)] if prev_year else pd.DataFrame()

#         prev_medals = np.nan
#         prev_gold_medals = np.nan

#         if not last_year_records.empty:
#             prev_medals = last_year_records['Medal'].count()
#             prev_gold_medals = last_year_records['Medal'][last_year_records['Medal'] == 'Gold'].count()

#         sports = series['Sport'].unique().tolist()
#         events = series['Event'].tolist()
#         total_medals = series['Medal'].count()
#         print(y,prev_medals,prev_gold_medals)
#         ds.append([noc,season,y,len(athletes),female_percentage,avg_age,avg_height,avg_weight,prev_medals,prev_gold_medals,len(sports),len(events),total_medals])
#     dataset.extend(ds)

In [ ]:
# 1. Separate seasons cleanly to avoid Summer/Winter crossover bugs
for season_name in ['Summer', 'Winter']:
    season_df = df[df['Season'] == season_name]
    sorted_years = np.sort(season_df['Year'].unique())
    
    for noc in nations:
        ds = []
        for i, y in enumerate(sorted_years):
            # Isolate this specific nation, year, and season
            series = season_df[(season_df['NOC'] == noc) & (season_df['Year'] == y)]
            
            if series.empty:
                continue
            
            # Unique athlete attributes
            athletes = series['ID'].unique().tolist()
            
            # Calculate female percentage correctly from unique athlete IDs
            unique_athletes = series.drop_duplicates(subset=['ID'])
            female_percentage = unique_athletes['Sex'].value_counts(normalize=True).get('F', 0.0)
            
            # Mean attributes
            avg_age = np.ceil(series['Age'].mean()) if not series['Age'].dropna().empty else np.nan
            avg_height = np.ceil(series['Height'].mean()) if not series['Height'].dropna().empty else np.nan
            avg_weight = np.ceil(series['Weight'].mean()) if not series['Weight'].dropna().empty else np.nan
            
            # True Unique Sports and Events counts
            sports = series['Sport'].nunique()
            events = series['Event'].nunique()
            
            # True Medal Counts (Team sports grouped safely by Event, Team, and Medal)
            medal_rows = series[series['Medal'].notna()]
            unique_medals = medal_rows.drop_duplicates(subset=['Event', 'Medal'])
            total_medals = len(unique_medals)
            
            # Extract previous year data within the SAME season
            prev_medals = np.nan
            prev_gold_medals = np.nan
            
            if i > 0:
                prev_year = sorted_years[i-1]
                last_year_records = season_df[(season_df['NOC'] == noc) & (season_df['Year'] == prev_year)]
                
                if not last_year_records.empty:
                    prev_medal_rows = last_year_records[last_year_records['Medal'].notna()]
                    unique_prev_medals = prev_medal_rows.drop_duplicates(subset=['Event', 'Medal'])
                    
                    prev_medals = len(unique_prev_medals)
                    prev_gold_medals = len(unique_prev_medals[unique_prev_medals['Medal'] == 'Gold'])
            
            # Append rows natively matching your historical schema
            ds.append([
                noc, season_name, y, len(athletes), female_percentage, 
                avg_age, avg_height, avg_weight, prev_medals, 
                prev_gold_medals, sports, events, total_medals
            ])
        dataset.extend(ds)


In [46]:
# features = df.groupby(["Year", "NOC"]).agg(
#     sports=("Sport", "nunique"),
#     athletes=("ID","nunique"),
#     avg_age=("Age","mean"),
#     events_entered=("Event", "count")
# )
# features
# pd.DataFrame(dataset,columns=new_cols).to_csv(OUT_DIR / "pre_clean.csv")
# pd.DataFrame(dataset,columns=new_cols).iloc[1347]
pd.DataFrame(dataset,columns=new_cols)

,noc,season,year,athletes,athletes_female_percentage,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals
0,CHN,Summer,1932,1,0.000000,22.0,175.0,70.0,NaN,NaN,1,2,0
1,CHN,Summer,1936,54,0.037037,24.0,175.0,69.0,0.0,0.0,7,27,0
2,CHN,Summer,1948,31,0.032258,26.0,175.0,70.0,0.0,0.0,6,12,0
3,CHN,Summer,1952,1,0.000000,23.0,175.0,70.0,0.0,0.0,1,1,0
4,CHN,Summer,1984,215,0.386047,23.0,173.0,66.0,NaN,NaN,19,105,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3832,MNE,Winter,2014,2,0.500000,19.0,180.0,73.0,0.0,0.0,1,3,0
3833,ALB,Winter,2006,1,0.000000,19.0,180.0,74.0,NaN,NaN,1,3,0
3834,ALB,Winter,2010,1,0.000000,23.0,180.0,74.0,0.0,0.0,1,2,0
3835,ALB,Winter,2014,1,1.000000,20.0,163.0,56.0,0.0,0.0,1,2,0
